Homework #5: model deployment

In [0]:
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
import mlflow
import mlflow.spark

In [0]:
catalog_name = "gr5069"
schema_name = "raw"

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

In [0]:
lr_table = "gr5069.raw.hw3122_lr_predictions"
rf_table = "gr5069.raw.hw3122_rf_predictions"

In [0]:
drivers = spark.read.option("header", True).option("inferSchema", True)\
    .csv("/Volumes/gr5069/raw/f1_data/drivers.csv")

results = spark.read.option("header", True).option("inferSchema", True)\
    .csv("/Volumes/gr5069/raw/f1_data/results.csv")

races = spark.read.option("header", True).option("inferSchema", True)\
    .csv("/Volumes/gr5069/raw/f1_data/races.csv")

constructors = spark.read.option("header", True).option("inferSchema", True)\
    .csv("/Volumes/gr5069/raw/f1_data/constructors.csv")

In [0]:
df = (
    results
    .join(races, "raceId", "left")
    .select(
        "resultId",
        "raceId",
        "driverId",
        "constructorId",
        "grid",
        "laps",
        "positionOrder",
        col("points").cast("double").alias("label")
    )
    .dropna()
)

df = df.withColumn(
    "started_from_pit",
    when(col("grid") == 0, 1).otherwise(0)
)

In [0]:
feature_cols = [
    "raceId",
    "driverId",
    "constructorId",
    "grid",
    "laps",
    "positionOrder",
    "started_from_pit"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

model_df = assembler.transform(df).select(
    "resultId",
    "raceId",
    "driverId",
    "constructorId",
    "features",
    "label"
)

In [0]:
train_df, test_df = model_df.randomSplit([0.8, 0.2], seed=42)

In [0]:
def evaluate_model(predictions):
    rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse").evaluate(predictions)
    mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae").evaluate(predictions)
    r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2").evaluate(predictions)
    mse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mse").evaluate(predictions)
    return rmse, mae, r2, mse

Model 1: Linear Regression

In [0]:
 import os

mlflow.set_experiment("/Users/hw3122@columbia.edu/F1_Points_hw3122")

with mlflow.start_run(run_name="Linear_Regression"):

    lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=20,
        regParam=0.1
    )

    model = lr.fit(train_df)
    preds = model.transform(test_df)

    rmse, mae, r2, mse = evaluate_model(preds)

    # log parameters
    mlflow.log_param("model", "LinearRegression")
    mlflow.log_param("maxIter", 20)
    mlflow.log_param("regParam", 0.1)

    # log metrics
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mse", mse)

    # artifact 1: metrics file
    metrics_path = "/tmp/lr_metrics.txt"
    with open(metrics_path, "w") as f:
        f.write(f"Linear Regression Metrics\n")
        f.write(f"RMSE: {rmse}\n")
        f.write(f"MAE: {mae}\n")
        f.write(f"R2: {r2}\n")
        f.write(f"MSE: {mse}\n")

    mlflow.log_artifact(metrics_path)

    # artifact 2: feature list file
    features_path = "/tmp/lr_features.txt"
    with open(features_path, "w") as f:
        f.write("Features used: " + ", ".join(feature_cols))

    mlflow.log_artifact(features_path)

    # create prediction output
    output = preds.select(
        "resultId",
        "raceId",
        "driverId",
        "constructorId",
        col("label").alias("actual_points"),
        col("prediction").alias("predicted_points")
    )

    # save predictions into table
output.createOrReplaceTempView("hw3122_lr_predictions")

In [0]:
spark.sql("SELECT * FROM hw3122_lr_predictions LIMIT 10").display()

Model 2: Random Forest

In [0]:
with mlflow.start_run(run_name="Random_Forest"):

    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol="label",
        numTrees=50,
        maxDepth=5
    )

    model = rf.fit(train_df)
    preds = model.transform(test_df)

    rmse, mae, r2, mse = evaluate_model(preds)

    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("numTrees", 50)
    mlflow.log_param("maxDepth", 5)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)
    mlflow.log_metric("mse", mse)

    # artifacts
    with open("/tmp/rf_metrics.txt", "w") as f:
        f.write(f"RMSE:{rmse}, MAE:{mae}")

    mlflow.log_artifact("/tmp/rf_metrics.txt")

    # prediction output
    output = preds.select(
        "resultId",
        "raceId",
        "driverId",
        "constructorId",
        col("label").alias("actual_points"),
        col("prediction").alias("predicted_points")
    )

    output.createOrReplaceTempView("hw3122_rf_predictions")

For this assignment, I used the F1 dataset to build two predictive models that predict the number of points a driver earns in a race.

The two models I built were:
1. Linear Regression
2. Random Forest Regressor

For each model, I logged hyperparameters, four evaluation metrics, and two artifacts in MLflow. The four metrics were RMSE, MAE, R2, and MSE. The two artifacts were a metrics text file and a feature list file.

I also generated prediction outputs for both models. The prediction outputs include resultId, raceId, driverId, constructorId, actual_points, and predicted_points.

I attempted to save the predictions into two permanent database tables using saveAsTable:

- gr5069.raw.hw3122_lr_predictions
- gr5069.raw.hw3122_rf_predictions

However, Databricks returned a permission error because my account does not have CREATE TABLE permission on the gr5069.raw schema. Because of this permission restriction, I stored the predictions as temporary views instead:

- hw3122_lr_predictions
- hw3122_rf_predictions

I successfully queried both temporary views to verify that the predictions were generated correctly. The MLflow runs were also successfully logged for both models.

In [0]:
spark.sql("SELECT * FROM hw3122_lr_predictions LIMIT 10").display()

In [0]:
spark.sql("SELECT * FROM hw3122_rf_predictions LIMIT 10").display()